In [1]:
# import
import psycopg2
import pandas as pd

import plotly.express as px

import math

In [2]:
# connect to postgresql
conn = psycopg2.connect(
    host = "localhost",
    database = "seismic",
    user = "postgres",
    password = "Codio590",
    port=5432)

print("Connected to Seismic")

Connected to Seismic


In [3]:
# verify connection / test
cur = conn.cursor()

cur.execute("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema='public'
""")

print(cur.fetchall())

[('sources',), ('earthquake_events',), ('locations',), ('earthquakes',)]


In [4]:
# Queries to populate dashboard

# pull info
    
query =  """
   SELECT 
       e.earthquake_id,
       e.magnitude,
       e.time,
       e.tsunami,
       e.significance,
       e.magnitude_type,
       e.title,
       l.place,
       l.latitude,
       l.longitude,
       l.depth_km,
       s.net,
       s.status
    FROM earthquake_events e
    LEFT JOIN locations l ON e.location_id = l.location_id
    LEFT JOIN sources s ON e.source_id = s.source_id
"""


df = pd.read_sql(query, conn)

# clean data
df["event_datetime"] = pd.to_datetime(df["time"], unit = "ms", errors = "coerce")
df["tsunami_label"] = df["tsunami"].map({0: "No Tsunami", 1: "Tsunami"})
df["magnitude_type"] = df["magnitude_type"].fillna("Unknown")

df.head()

/tmp/ipykernel_4620/1712883156.py:26: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,earthquake_id,magnitude,time,tsunami,significance,magnitude_type,title,place,latitude,longitude,depth_km,net,status,event_datetime,tsunami_label
0,us6000pzv4,4.7,1742427513049,0,340,mb,"M 4.7 - 93 km WNW of Pangai, Tonga","93 km WNW of Pangai, Tonga",-19.5417,-175.1973,137.1090,us,reviewed,2025-03-19 23:38:33.049,No Tsunami
1,tx2025gunm,1.7,1744037069049,0,44,ml,"M 1.7 - 54 km NW of Toyah, Texas","54 km NW of Toyah, Texas",31.6630,-104.1920,7.1765,tx,reviewed,2025-04-07 14:44:29.049,No Tsunami
2,tx2025gdoo,1.6,1743235103689,0,39,ml,"M 1.6 - 31 km NW of Toyah, Texas","31 km NW of Toyah, Texas",31.5310,-104.0090,6.8176,tx,reviewed,2025-03-29 07:58:23.689,No Tsunami
3,us7000pm7w,5.0,1742815298311,0,385,mb,"M 5.0 - 12 km SW of Xuyong, China","12 km SW of Xuyong, China",28.1045,105.3353,10.0000,us,reviewed,2025-03-24 11:21:38.311,No Tsunami
4,ak0254k0bylf,2.0,1744218069150,0,62,ml,"M 2.0 - 56 km NNE of Petersville, Alaska","56 km NNE of Petersville, Alaska",62.9554,-150.3107,81.6000,ak,automatic,2025-04-09 17:01:09.150,No Tsunami


In [5]:
# install dash
%pip install dash

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
from dash import Dash, dcc, html, Input, Output, dash_table

In [7]:
# create dashboard

app = Dash(__name__)

# grab the query without any duplicates and without blanks
mag_types = ["All"] + sorted(df["magnitude_type"].dropna().unique())

app.layout = html.Div([
    # add header
    html.H1("Earthquake Dashboard"),

    # add headers for magnitude and earthquakes
    html.Div([
        html.Div([
            html.H3("Total Earthquakes"),
            html.P(id = "total_count", style = {"fontSize": "22px"})]),
        html.Div([
            html.H3("Largest Magnitude"),
            html.P(id = "max_mag", style = {"fontSize": "22px"})]),
        html.Div([
            html.H3("Average Magnitude"),
            html.P(id = "avg_mag", style = {"fontSize": "22px"})])],
             style = {"display": "flex", "gap": "30px"}),
    
    # Add magnitude slider
    html.Label("Minimum Magnitude"),
    dcc.Slider(
        id = "mag_slider",
        min = float(df["magnitude"].min()),
        max = float(df["magnitude"].max()),
        step = 0.1,
        value = float(df["magnitude"].min()),
        marks = {i: str(i) for i in range(math.floor(df["magnitude"].min()), 
                                            math.ceil(df["magnitude"].max()) + 1)}),

    # Add magnitude type drop down
    html.Label("Magnitude Type"),
    dcc.Dropdown(
        id = "mag_type_dropdown",
        options = [{"label": m, "value": m} for m in mag_types],
        value = "All",
        clearable = False),

    # Add tsunami drop down
    html.Label("Tsunami"),
    dcc.Dropdown(
        id = "tsunami_dropdown",
        options = [{"label": "All", "value" : "All"},
                  {"label": "No", "value" : "No"},
                  {"label": "Yes", "value" : "Yes"}],
        value = "All",
        clearable = False),

    # add spacing
    html.Br(),
    
    # show graphs (histogram and map of earthquakes)
    dcc.Graph(id = "magnitude_hist"),
    dcc.Graph(id = "earthquake_map"),

    # data table for the dashboard
    dash_table.DataTable(
        id = "earthquake_table",
        columns = [
            {"name": "Place", "id": "place"},
            {"name": "Magnitude", "id": "magnitude"},
            {"name": "Depth KM", "id": "depth_km"},
            {"name": "Magnitude Type", "id": "magnitude_type"},
            {"name": "Tsunami", "id": "tsunami_label"},
            {"name": "Time (UTC)", "id": "event_datetime"}],
        page_size = 10,
        style_table = {"overflowX": "auto"},
        style_cell = {"textAlign": "left", "padding": "6px"},
        style_header = {"fontWeight": "bold"},
        sort_action = "native" )])
    

@app.callback(
    Output("total_count", "children"),
    Output("max_mag", "children"),
    Output("avg_mag", "children"),
    Output("magnitude_hist", "figure"),
    Output("earthquake_map", "figure"),
    Output("earthquake_table", "data"),
    Input("mag_slider", "value"),
    Input("mag_type_dropdown", "value"),
    Input("tsunami_dropdown", "value")
)

# update results for selected magnitudes/tsunami
def update_dash(min_mag, mag_type, tsunami_status):

    filtered = df[df["magnitude"] >= min_mag].copy()

    # if filtered on magnitude type, apply filter
    if mag_type != "All":
        filtered = filtered[filtered["magnitude_type"] == mag_type]

    # if filtered on whether or not there is a tsunami, apply filter
    if tsunami_status != "All":
        filtered = filtered[filtered["tsunami_label"] == tsunami_status]

    total_count = len(filtered)
    max_mag = round(filtered["magnitude"].max(), 2) if total_count > 0 else "N/A"
    avg_mag = round(filtered["magnitude"].mean(), 2) if total_count > 0 else "N/A"

    # create histogram
    hist_fig = px.histogram(
        filtered,
        x = "magnitude",
        title = "Earthquakes by Magnitude",
        labels = {"magnitude": "Magnitude"})

    # prepare map data
    df["latitude"] = pd.to_numeric(df["latitude"], errors = "coerce") 
    df["longitude"] = pd.to_numeric(df["longitude"], errors = "coerce")
    df["magnitude"] = pd.to_numeric(df["magnitude"], errors = "coerce")
    df["depth_km"] = pd.to_numeric(df["depth_km"], errors = "coerce")
    
    
    # add lat, long, magnitude to map data
    map_data = filtered.dropna(subset = ["latitude", "longitude", "magnitude"])

    # create map
    map_fig = px.scatter_geo(
        map_data,
        lat = "latitude",
        lon = "longitude",
        size = "magnitude",
        color = "tsunami_label",
        hover_name = "place",
        hover_data = ["magnitude", "depth_km", "magnitude_type", "event_datetime"],
        title = "Earthquake Locations")

    # edit whats displayed on map
    map_fig.update_geos(
        projection_type = "natural earth",
        showland = True,
        showcountries = True,
        showocean = False)

    # remove underscores from title and make text larger
    map_fig.update_layout(
        legend = dict(
            title = "Tsunami",
            font = dict(size = 16),
            title_font = dict(size = 18)))

    table_data = filtered[
        ["place", "magnitude", "depth_km", "magnitude_type", "tsunami_label", "event_datetime"]
        ].sort_values("magnitude", ascending = False).head(100).to_dict("records")

    return total_count, max_mag, avg_mag, hist_fig, map_fig, table_data
        

app.run(jupyter_mode = "inline", port = 8051)  

    